# Homework — Creating Region Objects in Python

**Course:** Medical Image Analysis & Processing (MIAP)

## Learning objectives

By the end of this notebook you should be able to:

1. Remove **high-frequency noise** with linear or median filtering.
2. Estimate a slowly varying **background** with a large median filter and use it for **flat-field correction**.
3. Apply **global thresholding** (Otsu) on a flat-field-corrected image.
4. Use `cv2.connectedComponentsWithStats` to obtain a labelled image together with per-region **statistics** (area, bounding box, centroid).
5. Inspect those statistics and **filter regions** with a logical rule that keeps the digits and discards the dashed frame and noise blobs.

## Input image

`tall_noise10_backgr.png` — densely packed digits inside a dashed-line frame, on a non-uniform background that also carries high-frequency noise.

## Expected final result

Your finished pipeline should produce the eight panels shown in the figure below — original ➜ denoised ➜ background ➜ flat-fielded ➜ thresholded ➜ labelled ➜ kept-only ➜ bounding boxes on the original.

In [ ]:
from IPython.display import Image, display
display(Image("expected_overview.png"))

---
## 0. Setup

Make sure the following packages are installed: `opencv-python`, `numpy`, `matplotlib`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 6)

IMAGE_PATH = "tall_noise10_backgr.png"

## Task 1 — Load and inspect the image

Read the image as **grayscale** with `cv2.imread(..., cv2.IMREAD_GRAYSCALE)` and print its shape, dtype, and intensity range. Show it with `plt.imshow(..., cmap="gray")`.

**Question 1.1.** What kind of degradations do you see? Identify at least three: low-frequency, high-frequency, and structural (frame).

*Low-frequency degradation: There is uneven illumination (shading) across the image. We can see noticeable shadows and darker regions, particularly near the edges and corners ,while other parts are brighter.*

*High-frequency degradation: The image exhibits visible noise or graininess, which creates a slightly speckled texture throughout both the background and the text.*

*Structural (frame) degradation: There is geometric distortion. The block of text is rotated, and there is noticeable perspective distortion or page curl, making the straight lines of text and the dashed bounding box appear curved and non-rectangular.*

**Question 1.2.** Why is it useful that the digits are clearly *darker* than the background?

*It is advantageous that the digits are darker than the background because it creates a clear intensity contrast between the objects of interest and the background. This has several benefits:*

*Simpler segmentation: Thresholding techniques can easily separate foreground (digits) from background because the intensity distributions are different.*

*Robust detection: Even with noise and uneven illumination, darker digits can still be distinguished from the brighter background.*

*Preprocessing flexibility: Methods such as adaptive thresholding, morphological operations, or background subtraction can exploit this contrast to isolate the digits more reliably.*

In [ ]:
# TODO: load the image, print shape/dtype/min/max, and display it.
img = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

print("Shape:", img.shape)
print("Data type:", img.dtype)
print("Min intensity:", img.min())
print("Max intensity:", img.max())

plt.imshow(img, cmap="gray")
plt.title("Input Image")
plt.axis("off")
plt.show()

## Task 2 — Remove high-frequency noise

Apply a small **median filter** (or a Gaussian if you prefer) to suppress the grainy noise. A `3 × 3` or `5 × 5` median is a good starting point.

**Hint.** Use `cv2.medianBlur(img, ksize)` — the kernel size must be odd.

**Question 2.1.** What happens to the digit strokes if you make the kernel too large?

If the median filter kernel is too large, it starts to blur or distort the digit strokes.
If the kernel size of a median filter is too large, the image will suffer from excessive smoothing and a significant loss of important high-frequency details. While a large kernel aggressively removes noise, it also treats fine textures, thin lines, and small structural elements (anything smaller than half the kernel’s area) as noise, completely washing them out. Furthermore, although median filters are generally good at preserving boundaries, an overly large kernel can cause edges to shift, sharp corners to become rounded, and the overall image to look overly heavily blurred , destroying essential structural information.


In [ ]:
# TODO: produce `denoised` from `img`.

#3x3 median filter
denoised3 = cv2.medianBlur(img, 3)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(denoised3, cmap="gray")
plt.title("Denoised(Median 3x3)")
plt.axis("off")

plt.tight_layout()
plt.show()


## Task 3 — Estimate the background

Run a **large** median filter on the denoised image. The kernel must be *bigger than the tallest object you want to remove* — i.e. larger than a digit — so that the median "forgets" the digits and only the slow background remains.

**Hint.** Try `ksize = 51`. Display the result; it should look like a smooth gray field.

**Question 3.1.** Why does mean / Gaussian filtering work less well here than median filtering for background estimation?

Mean and Gaussian filters are less effective for background estimation in this context because they function by averaging all pixel values within a neighborhood, meaning the dark digit strokes act as outliers that pull the average intensity downward. Consequently, instead of extracting a smooth background, these filters tend to produce an estimate that “shadows” or retains faint ghosts of the digits.

In contrast, the median filter is highly robust to such outliers; as long as the background pixels occupy more than half of the kernel’s area, the median operation effectively ignores the dark digit pixels and returns a clean, representative value from the background.

In [ ]:
# TODO: produce `background` from `denoised`.
background = cv2.medianBlur(denoised3, 51)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(denoised3, cmap="gray")
plt.title("Denoised(Median 3)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(background, cmap="gray")
plt.title("Background (Median 51)")
plt.axis("off")

plt.tight_layout()
plt.show()

## Task 4 — Flat-field correction

Subtract the background estimate from the denoised image so that what remains is (approximately) the digits on a uniform field. Because digits are *darker* than the background, compute `background − denoised` (use `cv2.subtract` to avoid uint8 underflow).

Display `diff` and verify visually that the slow brightness gradient is gone.

In [ ]:
# TODO: produce `diff`.

diff = cv2.subtract(background, denoised3)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(denoised3, cmap="gray")
plt.title("Denoised Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(diff, cmap="gray")
plt.title("diff")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Diff Min intensity:", diff.min())
print("Diff Max intensity:", diff.max())

## Task 5 — Global thresholding

Apply **Otsu's method** to the flat-fielded image to obtain a binary mask of the foreground.

**Hint.**
```python
thr_val, thresholded = cv2.threshold(
    diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
```

**Question 5.1.** Why would Otsu *fail* on the original image but work after flat-field correction?  

Otsu’s method would fail on the original image because the uneven illumination introduces a slow brightness gradient, meaning the background does not have a consistent intensity across the image. As a result, some background regions may be as dark as the digits in other regions, causing significant overlap between foreground and background intensity values in the histogram. Since Otsu’s algorithm assumes a bimodal histogram with clearly separable classes and selects a single global threshold, this overlap prevents it from finding a good separating value. After flat-field correction, however, the background becomes approximately uniform, and the digits stand out clearly in intensity. This produces a cleaner, more distinctly bimodal histogram, allowing Otsu’s method to successfully determine an effective global threshold.

**Question 5.2.** Inspect the binary image carefully. Are the digits well connected? Are there frame artefacts left? You will deal with those in Task 7.

After thresholding, the digits are generally well connected, showing up as clear white blobs against a black background. However, we might still notice some artefacts around the image border (frame) or small noisy spots.

In [ ]:
# TODO: compute Otsu threshold and binary image.
thr_val, thresholded = cv2.threshold(
    diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"Otsu threshold value: {thr_val}")

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(diff, cmap="gray")
plt.title("background - denoised")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(thresholded, cmap="gray")
plt.title(f"Otsu Thresholded ({thr_val})")
plt.axis("off")

plt.tight_layout()
plt.show()

## Task 6 — Connected components with statistics

Convert the binary image to a labelled image using `cv2.connectedComponentsWithStats`.

```python
connectivity = 8
num_labels, label_image, stats, centroids = \
    cv2.connectedComponentsWithStats(thresholded, connectivity, cv2.CV_32S)
```

Print:
* the number of labels (remember label `0` is the background),
* the columns of `stats`: `LEFT`, `TOP`, `WIDTH`, `HEIGHT`, `AREA` — accessible via the constants `cv2.CC_STAT_LEFT`, etc.

**Question 6.1.** Why might 4-connectivity and 8-connectivity give different counts here?

4-connectivity considers pixels connected only if they share an edge (up, down, left, or right), while 8-connectivity also treats diagonally touching pixels as connected.
In digit images, parts of a digit may touch only diagonally after thresholding. With 4-connectivity, such diagonal connections are not considered part of the same object, so a single digit may be split into multiple components, increasing the label count. With 8-connectivity, those diagonal pixels are treated as connected, so the digit remains a single component and the number of detected components is typically smaller.

In [ ]:
# TODO: run connectedComponentsWithStats and report the basic counts.

connectivity = 8

num_labels, label_image, stats, centroids = \
    cv2.connectedComponentsWithStats(thresholded, connectivity, cv2.CV_32S)

print("Number of labels (including background):", num_labels)
print("Background label: 0")

for i in range(num_labels):
    left = stats[i, cv2.CC_STAT_LEFT]
    top = stats[i, cv2.CC_STAT_TOP]
    width = stats[i, cv2.CC_STAT_WIDTH]
    height = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]

    print(f"Label {i}: LEFT={left}, TOP={top}, WIDTH={width}, HEIGHT={height}, AREA={area}")

## Task 7 — Investigate region properties

Plot histograms / scatter plots of the per-region statistics (skip label 0, the background).

Useful diagnostics:
* histogram of **area**,
* histogram of **height** (and width),
* scatter plot of **width vs. height**.

**Question 7.1.** From the histograms you should be able to see *two distinct populations*. What are they, and what physical objects in the image do they correspond to?  

The histograms reveal two distinct populations of connected components. One group consists of large components, which correspond to the actual digits in the image. These regions have larger areas and larger bounding box dimensions (width and height). The other group consists of very small components, which correspond to random noise blobs produced by thresholding or residual image noise. These components have very small areas and small widths and heights compared to the digits.

**Question 7.2.** Which single statistic best separates digits from the dashed frame? Which best separates digits from random noise blobs?

The area statistic is usually the best measure for separating digits from random noise blobs, because noise components contain very few pixels while digits contain significantly more. To separate digits from the dashed frame, the height (or width) of the bounding box works better, because the frame segments tend to be long and thin while digits occupy more balanced bounding boxes. The width–height relationship can also help distinguish these frame segments from digit shapes.

In [ ]:
# TODO: plot histograms / scatter of region statistics.

areas = stats[1:, cv2.CC_STAT_AREA]
widths = stats[1:, cv2.CC_STAT_WIDTH]
heights = stats[1:, cv2.CC_STAT_HEIGHT]

plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
plt.hist(areas, bins=20, color='steelblue', edgecolor='black')
plt.title("Histogram of Component Area")
plt.xlabel("Area (pixels)")
plt.ylabel("Count")

plt.subplot(2,2,2)
plt.hist(heights, bins=20, color='seagreen', edgecolor='black')
plt.title("Histogram of Component Height")
plt.xlabel("Height (pixels)")
plt.ylabel("Count")

plt.subplot(2,2,3)
plt.hist(widths, bins=20, color='orange', edgecolor='black')
plt.title("Histogram of Component Width")
plt.xlabel("Width (pixels)")
plt.ylabel("Count")

plt.subplot(2,2,4)
plt.scatter(widths, heights, c='purple')
plt.title("Width vs Height")
plt.xlabel("Width")
plt.ylabel("Height")

plt.tight_layout()
plt.show()


## Task 8 — Filter regions and build the segmented output

Combine your conditions with `np.logical_and` (or with the `&` operator) to obtain the labels you wish to keep, then build a binary image that contains *only* those regions.

The exercise sketches this exact pattern:

```python
keep_labels = np.where(np.logical_and(condition1, condition2, ...))
keep_label_image = np.isin(label_image, keep_labels).reshape(label_image.shape)
some_image[keep_label_image]        = 255
some_image[keep_label_image == False] = 0
```

> **Note.** `np.in1d` was used in older NumPy; in NumPy 2.x replace it with `np.isin`.

**Suggested rules** (start with these, then justify your own):
* `area >= 100`                — drop tiny noise blobs,
* `25 <= height <= 60`         — drop frame dashes (too short) and any blob that spans the frame (too tall),
* `5  <= width  <= 35`         — drop merged digit strings if any.

**Question 8.1.** What numerical justification can you give for each cut, based on the histograms from Task 7?

In [ ]:
# TODO: filter the labels and build the final segmented image.

areas   = stats[1:, cv2.CC_STAT_AREA]
widths  = stats[1:, cv2.CC_STAT_WIDTH]
heights = stats[1:, cv2.CC_STAT_HEIGHT]

condition_area   = areas >= 100
condition_height = np.logical_and(heights >= 25, heights <= 60)
condition_width  = np.logical_and(widths  >= 5,  widths  <= 35)

combined_condition = np.logical_and(condition_area,np.logical_and(condition_height, condition_width))

keep_labels = np.where(combined_condition)[0] + 1
keep_label_image = np.isin(label_image, keep_labels)
segmented = np.zeros_like(img)
segmented[keep_label_image] = 255

plt.imshow(segmented, cmap="gray")
plt.title("Final Segmented Digits")
plt.axis("off")
plt.show()

print("Number of Kept labels:", len(keep_labels))


## Task 9 — Visualise and verify

1. Show the segmented image (kept regions = white, everything else = black).
2. Draw the bounding boxes of the kept regions on the **original** image (`cv2.rectangle`) so you can verify visually that each box contains exactly one digit.
3. Build the 8-panel overview matching `expected_overview.png` so you can compare your pipeline to the reference.

**Question 9.1.** Roughly how many digits did you keep? Compare to the number of digits visible in the original image.  

The pipeline kept 788 connected components, while there are roughly 805 digits visible in the original image.

This means the segmentation worked very well overall and nearly all digits were correctly isolated as individual regions.

The small difference (about 2 % fewer components) suggests that a few digits were either merged together into a single component or slightly broken and lost during filtering, but the detection accuracy is still quite high.

**Question 9.2.** Where does your pipeline still fail? (e.g. digits split into two parts, two digits merged into one blob.) What pre-processing change might help?

The pipeline may still fail in cases where a digit is broken into multiple disconnected components, causing one digit to be counted as two or more regions. It may also fail if two nearby digits touch each other after thresholding, causing them to be merged into one connected component. Some frame artefacts or noise blobs may also remain if their area, width, and height are similar to real digits. A useful pre-processing improvement would be to adjust the median filter size used for background estimation or apply a small morphological operation after thresholding. For example, morphological closing could help reconnect broken digit strokes, while morphological opening could remove small noise blobs.


In [ ]:
# TODO: build and display the 8-panel overview figure.
plt.figure(figsize=(6, 8))
plt.imshow(segmented, cmap="gray")
plt.title("Segmented Image: Kept Regions Only")
plt.axis("off")
plt.show()

boxed = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

for label in keep_labels:
    x = stats[label, cv2.CC_STAT_LEFT]
    y = stats[label, cv2.CC_STAT_TOP]
    w = stats[label, cv2.CC_STAT_WIDTH]
    h = stats[label, cv2.CC_STAT_HEIGHT]

    cv2.rectangle(boxed,(x, y),(x + w, y + h),(0, 255, 0),2)

boxed_rgb = cv2.cvtColor(boxed, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 8))
plt.imshow(boxed_rgb)
plt.title("bounding Boxes on original")
plt.axis("off")
plt.show()

print("Number of kept digit regions:", len(keep_labels))

label_h, label_w = label_image.shape
color_label_image = np.zeros((label_h, label_w, 3), dtype=np.uint8)

num_labels = np.max(label_image)
colors = [np.random.randint(0, 255, 3).tolist() for i in range(num_labels + 1)]
colors[0] = [0, 0, 0] 

for label in range(1, num_labels + 1):
    color_label_image[label_image == label] = colors[label]

color_label_image = cv2.cvtColor(color_label_image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16, 12))

plt.subplot(2, 4, 1)
plt.imshow(img, cmap="gray")
plt.title("1. original")
plt.axis("off")

plt.subplot(2, 4, 2)
plt.imshow(denoised3, cmap="gray")
plt.title("2. denoised(Median 3)")
plt.axis("off")

plt.subplot(2, 4, 3)
plt.imshow(background, cmap="gray")
plt.title("3. background median(51)")
plt.axis("off")

plt.subplot(2, 4, 4)
plt.imshow(diff, cmap="gray")
plt.title("4. background - denoised")
plt.axis("off")

plt.subplot(2, 4, 5)
plt.imshow(thresholded, cmap="gray")
plt.title("5. Otsu Threshold")
plt.axis("off")

plt.subplot(2, 4, 6)
plt.imshow(color_label_image)
plt.title("6. Connected Components")
plt.axis("off")

plt.subplot(2, 4, 7)
plt.imshow(segmented, cmap="gray")
plt.title("7. Segmented Digits")
plt.axis("off")

plt.subplot(2, 4, 8)
plt.imshow(boxed_rgb)
plt.title("8. bounding Boxes on original")
plt.axis("off")

plt.tight_layout()
plt.show()


---
## Bonus tasks (optional)

* **B1.** Replace the global Otsu threshold with `cv2.adaptiveThreshold`. Does it remove the need for the background estimation step?

Adaptive thresholding reduces the dependence on background correction, but does not fully replace it for this image.

It improves robustness, but the uneven illumination in the dataset is too strong for adaptive thresholding alone.

Otsu tries to find one global threshold that fails with strong illumination gradients.Adaptive thresholding computes a threshold per local window, so it handles small lighting variations.

* **B2.** Compute additional region descriptors (perimeter, circularity, eccentricity, Hu moments) and plot them. Can you separate digits *by class* using only these features?

Some digits partially separate (e.g., “1” vs. “0”), but we cannot reliably classify all 10 digits with simple shape features alone.
These simple descriptors can cluster digits into loose groups, but not into all 10 digit classes.

A classifier (KNN, SVM, CNN) is needed for real classification.

* **B3.** Replace the median background with a **morphological top-hat** (`cv2.morphologyEx(..., cv2.MORPH_TOPHAT, ...)`) using a structuring element larger than a digit. Compare the two background estimates.

## What to hand in

* This notebook with all cells executed.
* Short written answers to **Questions 1.1 — 9.2** (markdown cells underneath each task are fine).
* A brief discussion (≤ 200 words) of the failure modes you observed and how you would address them.

In [ ]:
# B1

adaptive = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY_INV, 51, 10
)

plt.imshow(adaptive, cmap='gray')
plt.title("Adaptive Thresholding (No background sub)")
plt.show()


In [ ]:
#B2

areas = []
circularities = []

for label in keep_labels:
    mask = (label_image == label).astype(np.uint8)

    area = stats[label, cv2.CC_STAT_AREA]

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        continue

    perimeter = cv2.arcLength(contours[0], True)

    circ = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

    areas.append(area)
    circularities.append(circ)

plt.figure(figsize=(6, 5))
plt.scatter(areas, circularities, alpha=0.5)
plt.xlabel("Area")
plt.ylabel("Circularity")
plt.title("Area vs Circularity")
plt.grid(True)
plt.show()


In [ ]:
# B3

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (65, 65))

tophat = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel)

_, tophat_thresh = cv2.threshold(tophat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(tophat, cmap='gray')
plt.title("Top-Hat Result")
plt.subplot(1, 2, 2)
plt.imshow(tophat_thresh, cmap='gray')
plt.title("Otsu on Top-Hat")
plt.show()


$$Discussion$$
During the segmentation pipeline several failure modes were observed. First, some digits were broken into multiple components due to small gaps in strokes after thresholding. This caused over-counting in connected components(1111 instead 805). Second, a few neighbouring digits became merged into a single region, especially where strokes touched diagonally, leading to under-counting(788 instead of 805). Third, small residual artefacts from the frame and noise blobs occasionally survived the filtering rules. Finally, thin digits such as “1” were sometimes partially removed because strict width or area thresholds excluded them.

To address these issues, I would apply morphological closing before connected-component analysis to reconnect fragmented strokes, and morphological opening to suppress small noise blobs. Adjusting threshold parameters or using a slightly larger background estimation kernel could further stabilise binarisation. 

For merged digits, a distance transform with watershed segmentation could help separate touching components. Instead of fixed thresholds, adaptive filtering based on statistical clustering of region features (area, circularity) would make the system more robust across varying image conditions.